In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# 1) Add your PAT as a Kaggle secret: Settings → Secrets → "MY_GITHUB_TOKEN"
#    (must have repo + workflow scope so the pipeline can push artifacts)
os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("MY_GITHUB_TOKEN")

# 2) Fresh clone into the Kaggle working dir (NOT /content — that's Colab)
!rm -rf sports_prediction_model
!git clone -q https://github.com/andrewkemmer/sports_prediction_model.git sports_prediction_model

# 3) Install every dependency the NFL pipeline imports (backend, not the Streamlit frontend)
!pip install -q nflreadpy polars scikit-learn lightgbm xgboost \
    pandas numpy joblib gitpython requests
print("setup done")import os

# --- OPTIONAL overrides (omit everything for a normal daily run) ---

# Slate target season: the CURRENT schedule the games[] slate stage predicts
# (2026 week 1). Omit = auto-detect from the loaded schedule.
# os.environ["NFL_SLATE_SEASON"] = "2026"

print("run options set")import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"
cmd = ["python", "nfl-backend/backend/master_pipeline.py"]

result = subprocess.run(cmd, cwd=repo, env=os.environ.copy(), capture_output=False)

# The pipeline's Phase 5 already pushes nfl-backend/data_delivery/ to GitHub
# itself. Fail loudly if it didn't complete — Kaggle marks the run failed.
if result.returncode != 0:
    raise SystemExit(f"Pipeline failed with exit code {result.returncode}")
print("Pipeline completed — artifacts pushed to GitHub by Phase 5 sync.")import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"

# Confirm main moved: fetch and compare HEAD to the pre-run HEAD
check = subprocess.run(
    ["git", "log", "--oneline", "-1"],
    cwd=repo, capture_output=True, text=True,
)
print("Repo HEAD after run:", check.stdout.strip())

# Sanity: newest dated NFL artifact on main
ls = subprocess.run(
    ["git", "ls-tree", "-r", "--name-only", "origin/main"],
    cwd=repo, capture_output=True, text=True,
)
nfl_artifacts = [f for f in ls.stdout.splitlines()
                 if "nfl-backend/data_delivery/" in f and "_2026" in f]
print("Latest NFL artifacts:", sorted(nfl_artifacts)[-4:] if nfl_artifacts else "none found")